<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# Section 1: Ranked Actions & Reason Codes Engine

### Queue Prioritization Rules
* **Primary Ranking Signal:** URLs are strictly ordered based on **GSC Impression Decay (`gsc_impressions`)**—pages with the lowest or most rapidly declining impression volumes are prioritized at the top of the queue.
* **Tie-Breaker Rule:** If two or more URLs share identical impression values, the queue resolves ties using **`client_id` in chronological order**.
* **Core Rationale:** Impression drops serve as the earliest leading indicator of organic visibility decay before click and ranking drops fully materialize.

---

### Priority Queue & Reason Code Mapping

| Priority Rank | Trigger Criteria | Assigned Reason Code | Required Human Action |
| :--- | :--- | :--- | :--- |
| **P1 — Critical** | Lowest GSC Impressions + High-Confidence Model DOWN Signal | `R1_SEVERE_IMPRESSION_DECAY` | Immediate content refresh & technical SEO audit |
| **P2 — High** | Moderate Impression Drop + DOWN Signal | `R2_TRAFFIC_MOMENTUM_LOSS` | Keyword re-optimization & SERP intent check |
| **P3 — Medium** | Volatile/Flat Impressions + Low Model Confidence | `R3_CTR_ENGAGEMENT_DROP` | Title/Meta update & internal linking boost |
| **P4 — Low** | Stable Impressions (FLAT / UP trajectories) | `R4_STABLE_TRAJECTORY` | Standard routine monitoring |

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

# Section 2: Intended Use, Scope & Operational Boundaries

### Target Audience & Core Purpose
* **Primary Users:** In-house SEO Strategists, Content Editors, and Digital Marketing Teams at FlyRank client organizations.
* **Intended Use Case:** Serving as an **Early-Warning Predictive Advisory System** to identify declining URLs, prioritize content optimization queues, and provide actionable reason codes to streamline manual SEO investigations.

---

### System Boundaries & Operational Limits

* **No Autonomous Action:** The system is purely decision-support; it generates recommendation queues and alert flags, **never** executing automated content modifications, canonical changes, or site updates directly.
* **Geographic & Search Scope:** Optimized specifically for organic Google Search trajectories; invalid for non-search acquisition channels (e.g., direct traffic, paid ads, social campaigns) or alternative search engines.
* **Temporal Drift Cutoff:** Valid for short-to-medium horizon forecasting ($30\text{–}90\text{ day}$ windows). Model reliability degrades past 90 days due to market dynamics and non-stationary search engine algorithms.

---

### Explicit No-Go Automation Scenarios

1. **Abrupt Search Engine Algorithm Updates:** During broad Google core updates or major SERP structural shifts, standard historical signals become temporarily invalid; human review is mandatory.
2. **High-Revenue & Brand-Core URLs:** Key revenue-generating landing pages, core brand assets, and homepage URLs are strictly exempt from automated queue triage and require expert manual audit.
3. **Low-Volume Volatile Pages:** Long-tail or newly launched pages with minimal historical baseline impressions display high variance where percentage changes do not reflect true organic decay.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

# Section 3: Human-in-the-Loop Protocol & Operational No-Go List

### Pre-Action Human Verification Checklist
Before executing any optimization action recommended by the queue, an SEO analyst must manually verify:

* **SERP Layout & Feature Intent:** Confirm if impression loss is driven by SERP feature changes (e.g., AI Overviews, Featured Snippets, PAA blocks) rather than organic ranking loss.
* **Competitor Content Changes:** Inspect top-ranking competitors for recent content updates, technical structural improvements, or new backlink acquisitions.
* **Technical Infrastructure & On-Page Health:** Rule out site-level technical glitches, broken internal links, accidental `noindex` tags, or server performance degradation.
* **Real-World Business Relevance:** Verify that the flagged URL remains aligned with current business goals, active product inventory, or seasonal relevance.

---

### Strict No-Go Automation List (Automated Actions BANNED)

1. **Direct Content Rewriting or Auto-Publishing:** AI-generated text or automated content updates must **never** be auto-published to live client sites without editorial oversight.
2. **Indexing & Structural Directives:** Automatic application of `noindex` tags, URL redirects, canonical changes, or page deletions is strictly prohibited.
3. **Core Brand & Enterprise Revenue Pages:** Top-converting landing pages, core product catalog templates, and homepages must bypass automated workflows and undergo direct manual review.
4. **Volatile Market & Algorithm Shifts:** Automated execution is frozen during confirmed Google Core Updates, broad SERP layout redesigns, or sudden brand reputation events.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

# Section 4: Monitoring Framework & Model Retrain Triggers

### Performance Degradation Metrics
Recommendations are flagged as stale and requiring review if operational metrics cross the following thresholds:

* **Macro F1 Score Drop:** Model Macro F1 falls below **$40.0\%$** on rolling 30-day evaluation windows (Baseline: $45.36\%$).
* **DOWN Class Precision Decay:** Precision for `DOWN` predictions drops below **$65.0\%$** (Baseline: $72.25\%$), indicating an unacceptable rise in false alarm alerts.
* **Severe Error Rate Spike:** Cross-directional severe errors ($\text{DOWN} \leftrightarrow \text{UP}$) exceed **$22.0\%$** of total predictions (Baseline: $18.24\%$).

---

### Environmental & System Retrain Triggers

* **Data & Feature Drift Threshold:** Population Stability Index ($\text{PSI}$) exceeds **$0.25$** on key input features (`gsc_impressions`, momentum deltas) compared to the 2025 training baseline.
* **Search Engine Algorithm Updates:** Automatic retrain pipeline is triggered **14 to 21 days following a confirmed Google Core Update** to capture recalibrated SERP feature behaviors.
* **Cadence-Based Scheduled Retraining:** Mandatory quarterly retrain cycle (every **90 days**) using the most recent rolling 12-month temporal dataset to maintain temporal validity.
* **Human Feedback Loop Threshold:** SEO analyst disagreement rate with P1/P2 queue recommendations exceeds **$25.0\%$** over a continuous 14-day window.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**FINAL FLYRANK PREDICTION QUEUE GENERATION**

In [5]:
# =============================================================================
# ML-09 / NOTEBOOK 7
# FINAL DOWN-ONLY PREDICTION QUEUE GENERATION WITH TARGET COLUMN
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

print("=" * 100)
print("FINAL FLYRANK PREDICTION QUEUE GENERATION (STRICTLY DOWN ROWS)")
print("=" * 100)

# 1. LOAD DATASET
DATA_PATH = "/content/finalest90drollingwindow.parquet"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Parquet file not found at: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH).copy()

required = ["target", "window_start"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
df = df.dropna(subset=["window_start", "target"]).copy()
df["target"] = pd.to_numeric(df["target"], errors="coerce").dropna().astype(int)

# 2. LOCK JAN-2026 TEST SET & BALANCED TRAIN SET
train_pool = df[df["window_start"] < "2026-01-01"].copy()
test_df = df[df["window_start"] >= "2026-01-01"].copy()

if len(test_df) == 0:
    raise ValueError("No Jan-2026 test rows found.")

BALANCED_MONTHS = ["2025-04", "2025-05", "2025-06", "2025-07", "2025-08"]
train_pool["train_month"] = train_pool["window_start"].dt.to_period("M").astype(str)
balanced_train = train_pool[train_pool["train_month"].isin(BALANCED_MONTHS)].copy()

# Class Balancing
class_counts = balanced_train["target"].value_counts()
min_class_size = int(class_counts.min())
balanced_parts = [
    balanced_train[balanced_train["target"] == cls].sample(n=min_class_size, random_state=42)
    for cls in [0, 1, 2]
]
balanced_train = pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# 3. FEATURE SELECTION & TRAINING
EXPLICIT_EXCLUDE = {
    "target", "window_start", "window_end", "future_start", "future_end",
    "future_imp_3m", "future_impression_change_pct", "trend_direction_future"
}

feature_candidates = [
    col for col in balanced_train.columns
    if col not in EXPLICIT_EXCLUDE and not any(k in str(col).lower() for k in ["future_", "target", "label"])
    and str(col).lower() not in {"content_id", "client_id", "url", "page_url"}
]

numeric_features = [col for col in feature_candidates if pd.api.types.is_numeric_dtype(balanced_train[col])]

X_train = balanced_train[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y_train = balanced_train["target"].copy()
X_test = test_df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

train_medians = X_train.median()
X_train = X_train.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

model = RandomForestClassifier(n_estimators=150, max_depth=18, min_samples_leaf=2, max_features="sqrt", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 4. PREDICTIONS & DECAY SCORING
pred = model.predict(X_test)
probabilities = model.predict_proba(X_test)

queue = test_df.copy()
queue["predicted_target"] = pred
queue["best_model_name"] = "Balanced Random Forest"
queue["best_model_probability"] = probabilities.max(axis=1)
queue["confidence"] = queue["best_model_probability"]

down_prob_idx = list(model.classes_).index(0) if 0 in model.classes_ else 0
queue["down_probability"] = probabilities[:, down_prob_idx]

# Extract GSC Impression Column
imp_col = None
for c in ["impressions_90d", "gsc_impressions_mean_3m", "gsc_impressions_last", "gsc_impressions"]:
    if c in queue.columns:
        imp_col = c
        break

if imp_col is not None:
    queue["raw_impressions"] = pd.to_numeric(queue[imp_col], errors="coerce").fillna(0)
else:
    queue["raw_impressions"] = 0

# SCORING LOGIC: Low/Zero Impressions = High Decay Rate Score (0 Impression -> 1.0 Score)
max_imp = queue["raw_impressions"].max() if queue["raw_impressions"].max() > 0 else 1.0
queue["baseline_refresh_score"] = 1.0 - (queue["raw_impressions"] / max_imp)
queue["final_refresh_score"] = (0.70 * queue["down_probability"]) + (0.30 * queue["baseline_refresh_score"])

# 5. STRICT FILTER: FILTER ONLY DOWN (0) PREDICTIONS & EXCLUDE UP/FLAT
down_queue = queue[queue["predicted_target"] == 0].copy()

# ADD REQUIRED TARGET DISPLAY COLUMN
down_queue["target"] = "DOWN (0)"
down_queue["is_declining_label"] = 1
down_queue["suggested_action"] = "PRIORITIZE_REFRESH"
down_queue["final_reason_codes"] = "R1_SEVERE_IMPRESSION_DECAY"

# Helper function to map or safely initialize sample columns
def map_or_default(df_in, candidates, default_val=np.nan):
    for c in candidates:
        if c in df_in.columns:
            return df_in[c]
    return default_val

down_queue["content_id"] = map_or_default(down_queue, ["content_id", "url", "page_url"], default_val=down_queue.index)
down_queue["client_id"] = map_or_default(down_queue, ["client_id", "website_id"], default_val="CLIENT_001")
down_queue["impressions_90d"] = down_queue["raw_impressions"]
down_queue["clicks_90d"] = map_or_default(down_queue, ["clicks_90d", "gsc_clicks_mean_3m", "gsc_clicks_last"])
down_queue["sessions_90d"] = map_or_default(down_queue, ["sessions_90d", "sessions"])
down_queue["avg_position"] = map_or_default(down_queue, ["avg_position", "gsc_position_mean_3m"])
down_queue["ctr"] = map_or_default(down_queue, ["ctr", "gsc_ctr_mean_3m"])
down_queue["content_age_days"] = map_or_default(down_queue, ["content_age_days", "age_days"])
down_queue["days_since_last_update"] = map_or_default(down_queue, ["days_since_last_update"])
down_queue["word_count"] = map_or_default(down_queue, ["word_count"])
down_queue["trend_direction"] = map_or_default(down_queue, ["trend_direction", "impression_momentum"])
down_queue["competition_level"] = map_or_default(down_queue, ["competition_level"])
down_queue["content_type"] = map_or_default(down_queue, ["content_type"])
down_queue["main_intent"] = map_or_default(down_queue, ["main_intent", "search_intent"])
down_queue["age_tier"] = map_or_default(down_queue, ["age_tier"])
down_queue["freshness_tier"] = map_or_default(down_queue, ["freshness_tier"])
down_queue["word_count_tier"] = map_or_default(down_queue, ["word_count_tier"])
down_queue["impression_tier"] = map_or_default(down_queue, ["impression_tier"])
down_queue["position_tier"] = map_or_default(down_queue, ["position_tier"])

# 6. SORTING DOWN QUEUE (HIGHEST DECAY SCORE -> CLIENT_ID TIE-BREAKER)
down_queue = down_queue.sort_values(
    by=["baseline_refresh_score", "client_id"],
    ascending=[False, True]
).reset_index(drop=True)

# Generate strict sequential rank for DOWN rows only (1 to N)
down_queue["final_rank"] = np.arange(len(down_queue)) + 1

# 7. EXACT SCHEMA INCL. 'target' COLUMN
FINAL_COLUMNS = [
    "final_rank", "target", "content_id", "client_id", "final_refresh_score", "best_model_name",
    "best_model_probability", "baseline_refresh_score", "confidence", "suggested_action",
    "final_reason_codes", "is_declining_label", "impressions_90d", "clicks_90d",
    "sessions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update",
    "word_count", "trend_direction", "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

final_queue = down_queue[FINAL_COLUMNS].copy()

# 8. EXPORT
OUTPUT_DIR = "/content/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "flyrank_prediction_queue.csv")

final_queue.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 100)
print("FINAL DOWN PREDICTION QUEUE EXPORT COMPLETE")
print("=" * 100)
print(f"Exported file : {OUTPUT_PATH}")
print(f"Total DOWN Rows: {len(final_queue):,}")
print(f"Total Columns  : {len(final_queue.columns)}")

print("\nTOP 5 DOWN PRIORITY QUEUE ROWS:")
print("=" * 100)
print(final_queue.head(5).to_string(index=False))
print("=" * 100)

FINAL FLYRANK PREDICTION QUEUE GENERATION (STRICTLY DOWN ROWS)

FINAL DOWN PREDICTION QUEUE EXPORT COMPLETE
Exported file : /content/work/outputs/flyrank_prediction_queue.csv
Total DOWN Rows: 32,964
Total Columns  : 29

TOP 5 DOWN PRIORITY QUEUE ROWS:
 final_rank   target  content_id  client_id  final_refresh_score        best_model_name  best_model_probability  baseline_refresh_score  confidence   suggested_action         final_reason_codes  is_declining_label  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count  trend_direction  competition_level  content_type  main_intent  age_tier  freshness_tier  word_count_tier  impression_tier  position_tier
          1 DOWN (0)        1209 CLIENT_001             0.771456 Balanced Random Forest                0.673509                0.999999    0.673509 PRIORITIZE_REFRESH R1_SEVERE_IMPRESSION_DECAY                   1         0.333333         0.0           NaN           NaN  NaN     

In [6]:
print(final_queue.tail(10).to_string(index=False))

 final_rank   target  content_id  client_id  final_refresh_score        best_model_name  best_model_probability  baseline_refresh_score  confidence   suggested_action         final_reason_codes  is_declining_label  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count  trend_direction  competition_level  content_type  main_intent  age_tier  freshness_tier  word_count_tier  impression_tier  position_tier
      32955 DOWN (0)      507657 CLIENT_001             0.822144 Balanced Random Forest                0.932167                0.565426    0.932167 PRIORITIZE_REFRESH R1_SEVERE_IMPRESSION_DECAY                   1    120274.000000  262.000000           NaN           NaN  NaN               NaN                     NaN         NaN              NaN                NaN           NaN          NaN       NaN             NaN              NaN              NaN            NaN
      32956 DOWN (0)      346862 CLIENT_001             0.482221

In [7]:
# 8. Export CSV File
EXACT_COLUMNS = [
    "final_rank", "target", "content_id", "client_id", "final_refresh_score", "best_model_name",
    "best_model_probability", "baseline_refresh_score", "confidence", "suggested_action",
    "final_reason_codes", "is_declining_label", "impressions_90d", "clicks_90d",
    "sessions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update",
    "word_count", "trend_direction", "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

OUTPUT_DIR = "/content/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "flyrank_prediction_queue.csv")

# Generate CSV
down_queue[EXACT_COLUMNS].to_csv(OUTPUT_PATH, index=False)
print(f"CSV file exported successfully to: {OUTPUT_PATH}")

CSV file exported successfully to: /content/work/outputs/flyrank_prediction_queue.csv


In [9]:
# Save DOWN-only queue directly as Parquet
OUTPUT_PATH = "/content/work/outputs/flyrank_prediction_queue.parquet"
down_queue[EXACT_COLUMNS].to_parquet(OUTPUT_PATH, index=False)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.